# Kaitu AI Brain Fine-Tuning: Phi-4-mini with QLoRA
This notebook fine-tunes **microsoft/Phi-4-mini-instruct (3.8B)** using 4-bit QLoRA and PEFT.
It is designed to run completely within a **Free Google Colab T4 GPU** (15 GB VRAM) or **Kaggle P100**.

In [ ]:
# 1. GPU Check
!nvidia-smi

In [ ]:
# 2. Install Required Dependencies
!pip install -q -U torch transformers datasets peft trl bitsandbytes accelerate

In [ ]:
# 3. Upload or Load Dataset
# You can upload train.jsonl and eval.jsonl directly to the Colab files pane
import os
from datasets import load_dataset

train_file = 'train.jsonl'
eval_file = 'eval.jsonl'

if not os.path.exists(train_file):
    print(f'Please upload {train_file} and {eval_file} to this Colab workspace.')
else:
    dataset = load_dataset('json', data_files={'train': train_file, 'eval': eval_file})
    print(f'Train samples: {len(dataset["train"])}, Eval samples: {len(dataset["eval"])}')

In [ ]:
# 4. Load Base Model in 4-bit NF4 Quantization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = 'microsoft/Phi-4-mini-instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 5. Train with TRL SFTTrainer
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='./kaitu-checkpoints',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    num_train_epochs=3,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    logging_steps=10,
    save_strategy='epoch',
    evaluation_strategy='epoch',
    bf16=True,
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['eval'],
    peft_config=lora_config,
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=1024
)

trainer.train()

In [ ]:
# 6. Save LoRA Adapter
adapter_path = './kaitu-phi4-mini-adapter'
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'Adapter saved to {adapter_path}')

In [ ]:
# 7. Merge LoRA into Base Model
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map='cpu',
    trust_remote_code=True
)
merged_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = merged_model.merge_and_unload()

merged_out = './kaitu-phi4-mini-merged'
merged_model.save_pretrained(merged_out)
tokenizer.save_pretrained(merged_out)
print(f'Merged model saved to {merged_out}')

In [ ]:
# 8. Convert to GGUF using llama.cpp and quantize to Q4_K_M
!git clone https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp && pip install -r requirements.txt
!python llama.cpp/convert_hf_to_gguf.py ./kaitu-phi4-mini-merged --outfile ./kaitu-phi4-mini-f16.gguf --outtype f16
!./llama.cpp/llama-quantize ./kaitu-phi4-mini-f16.gguf ./kaitu-phi4-mini-q4_k_m.gguf Q4_K_M
print('Quantized GGUF artifact ready: kaitu-phi4-mini-q4_k_m.gguf')